# 📊 Exploration du Dataset IMDb — Groupe G02
**Problématique P02 : Régularisation et Généralisation**  
**Modèle : BERT-base-uncased | Dataset : IMDb (D01)**

Ce notebook explore le dataset IMDb avant l'entraînement :
- Distribution des classes
- Distribution des longueurs
- Analyse de la tokenisation BERT
- Exemples représentatifs

## 0. Installation et imports

In [ ]:
# Installation des dépendances (Colab)
import os
if not os.path.exists('/content/G02_PROJET'):
    raise FileNotFoundError('Clonez le dépôt dans /content/G02_PROJET avant de lancer ce notebook.')

!pip install -r /content/G02_PROJET/requirements.txt -q
print('✅ Dépendances installées')

In [ ]:
import sys
sys.path.insert(0, '/content/G02_PROJET/src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from data_loader import load_imdb_subset, explore_dataset, analyze_tokenization
from config import SEED, FIGURES_DIR

print('✅ Imports OK')

## 1. Chargement du dataset

In [ ]:
subsets = load_imdb_subset(
    num_train_per_class=300,
    num_val_per_class=100,
    num_test_per_class=150,
    seed=SEED,
    verbose=True
)
print(f"\nDataset chargé : {sum(len(v) for v in subsets.values())} exemples au total")

## 2. Distribution des classes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Distribution des classes — IMDb (G02)', fontsize=13, fontweight='bold')

for ax, (split, data) in zip(axes, subsets.items()):
    labels = [ex['label'] for ex in data]
    counts = Counter(labels)
    ax.bar(['Négatif (0)', 'Positif (1)'],
           [counts[0], counts[1]],
           color=['#E74C3C', '#2ECC71'], edgecolor='black', linewidth=0.5)
    ax.set_title(f'{split} ({len(data)} ex.)')
    ax.set_ylabel('Nombre d\'exemples')
    for i, (k, v) in enumerate(sorted(counts.items())):
        ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
    ax.set_ylim(0, max(counts.values()) * 1.2)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/explore_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dataset parfaitement équilibré (50/50 pos/neg)')

## 3. Distribution des longueurs

In [ ]:
stats = explore_dataset(subsets, max_length=200, save=True)
plt.show()

## 4. Analyse de la tokenisation BERT

In [ ]:
analyze_tokenization(subsets, n_samples=10)

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('bert-base-uncased')

ratios = []
for ex in subsets['train']:
    words  = len(ex['text'].split())
    tokens = len(tok.encode(ex['text'], add_special_tokens=False))
    ratios.append(tokens / max(words, 1))

print(f'Ratio mots→tokens moyen : {np.mean(ratios):.3f}')
print(f'Max_length=200 tokens ≈ {200/np.mean(ratios):.0f} mots réels')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ratios, bins=30, color='#3498DB', edgecolor='white', alpha=0.8)
ax.axvline(np.mean(ratios), color='red', linestyle='--', label=f'Moyenne = {np.mean(ratios):.3f}')
ax.set_xlabel('Ratio tokens/mots')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution du ratio de tokenisation BERT (WordPiece)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/explore_tokenization_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Exemples représentatifs

In [ ]:
print('=== 3 critiques POSITIVES ===\n')
pos_examples = [ex for ex in subsets['train'] if ex['label'] == 1][:3]
for i, ex in enumerate(pos_examples, 1):
    print(f'[{i}] {ex["text"][:200]}...\n')

print('\n=== 3 critiques NÉGATIVES ===\n')
neg_examples = [ex for ex in subsets['train'] if ex['label'] == 0][:3]
for i, ex in enumerate(neg_examples, 1):
    print(f'[{i}] {ex["text"][:200]}...\n')

## 6. Synthèse

| Caractéristique | Valeur |
|---|---|
| Classes | 2 (pos/nég) — parfaitement équilibrées |
| Train / Val / Test | 600 / 200 / 300 |
| Longueur moy. (mots) | ~235 mots |
| Ratio tokenisation | ~1.24 tokens/mot |
| max_length=200 tokens | ≈ 161 mots réels |
| Exemples tronqués | ~42% du train |